# RuleShift Notebook

## Runtime bootstrap

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

RUNTIME_SRC = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime/src")
if not RUNTIME_SRC.is_dir():
    raise FileNotFoundError(
        "Runtime dataset not found at /kaggle/input/datasets/raptorengineer/ruleshift-runtime/src — "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

## Frozen leaderboard split

In [ ]:
from core.kaggle import (
    build_kaggle_payload,
    load_leaderboard_dataframe,
    normalize_count_result_df,
    run_binary_task,
    validate_kaggle_payload,
)
PRIVATE_DATASET_ROOT, frozen_splits, leaderboard_df = load_leaderboard_dataframe()

## Pre-run status

In [ ]:
import pandas as pd
from core.splits import MANIFEST_VERSION

_splits_loaded = sorted(frozen_splits.keys())
_episode_count = len(leaderboard_df)
_private_attached = PRIVATE_DATASET_ROOT is not None

_status_df = pd.DataFrame(
    [
        ("Benchmark version", MANIFEST_VERSION),
        ("Episodes loaded", _episode_count),
        ("Splits available", ', '.join(_splits_loaded)),
        ("Private dataset", 'yes' if _private_attached else 'no'),
    ],
    columns=["Field", "Value"],
)

_status_df.style.hide(axis="index").set_caption("RuleShift pre-run status")

## Official task

In [ ]:
_RULESHIFT_BINARY_DF = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary_row",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
    store_task=False,
)
def _ruleshift_benchmark_v1_binary_row(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    del split
    del episode_id
    return run_binary_task(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
    )


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> dict:
    import json

    global _RULESHIFT_BINARY_DF

    eval_df = leaderboard_df[["episode_id", "split", "prompt_binary", "probe_targets"]].copy()
    binary_results = _ruleshift_benchmark_v1_binary_row.evaluate(
        llm=[llm],
        evaluation_data=eval_df,
    )
    binary_df = normalize_count_result_df(
        binary_results.as_dataframe(),
        dataframe_label="binary_df",
    )
    payload = build_kaggle_payload(binary_df=binary_df)
    validate_kaggle_payload(payload)

    _RULESHIFT_BINARY_DF = binary_df

    print("=== RuleShift Notebook — Canonical Result ===")
    print(json.dumps(payload, indent=2))
    print("=== End Canonical Result ===")
    return payload


## Official payload

In [ ]:
payload = ruleshift_benchmark_v1_binary(kbench.llm)

## Post-run result

In [ ]:
_score_pct = f"{payload['score'] * 100:.1f}%"
_num = payload["numerator"]
_den = payload["denominator"]
_episodes = payload["total_episodes"]
_version = payload["benchmark_version"]

_result_df = pd.DataFrame(
    [
        ("Final score", _score_pct),
        ("Correct / Total probes", f"{_num} / {_den}"),
        ("Episodes evaluated", _episodes),
        ("Payload validation", "Passed"),
        ("Benchmark version", _version),
    ],
    columns=["Field", "Value"],
)

_result_df.style.hide(axis="index").set_caption("RuleShift post-run result")

## Analytical summary

In [ ]:
_bdf = _RULESHIFT_BINARY_DF
_has_misses = (
    _bdf is not None
    and {'num_correct', 'total'}.issubset(_bdf.columns)
    and (_bdf['num_correct'] < _bdf['total']).any()
)

if _has_misses:
    _m = _bdf[_bdf['num_correct'] < _bdf['total']].copy()
    _id_cols = [c for c in ['episode_id', 'split'] if c in _m.columns]
    _summary_df = _m.loc[:, _id_cols].copy()
    _summary_df['correct'] = _m['num_correct'].astype(int)
    _summary_df['total'] = _m['total'].astype(int)
    _summary_df['missed'] = (_m['total'] - _m['num_correct']).astype(int)
    _title = f"RuleShift episodes with misses ({len(_summary_df)})"
else:
    _title = "RuleShift benchmark composition"
    _rows = []
    for _sn, _feps in frozen_splits.items():
        for _fse in _feps:
            _rows.append((_sn.replace('_leaderboard', ''), _fse.episode.difficulty.name.lower()))
    _comp = pd.DataFrame(_rows, columns=['split', 'difficulty'])
    _summary_df = _comp.groupby('split')['difficulty'].value_counts().unstack(fill_value=0)
    _order = [c for c in ['easy', 'medium', 'hard'] if c in _summary_df.columns]
    _summary_df = _summary_df[[*_order, *[c for c in _summary_df.columns if c not in _order]]]
    _summary_df['total'] = _summary_df.sum(axis=1)
    _summary_df = _summary_df.reset_index()

_summary_df.style.hide(axis="index").set_caption(_title)

## Task selection

In [ ]:
%choose ruleshift_benchmark_v1_binary